# The mask

A request asks several questions about one context. The model reads the context once and answers every
question in the same pass, and yet no question can influence another. What makes that possible is the
attention mask, built by `branch_mask`.

This page is a notebook. The reference entry below is pulled from the source, and the grids are what the
code printed when the notebook was last run.

In [ ]:
#| echo: false
from fastcore.docments import MarkdownRenderer as show_doc

from babykev.model import branch_mask

show_doc(branch_mask)

```python
def branch_mask(
    seg:list[int], # The segment of each token: 0 for the state, k for question k
    device:str, # Where to build the mask
    dtype:torch.dtype=torch.float32, # The number type of the model's attention scores
)->torch.Tensor: # Shape [1, 1, n, n]: 0 where i may attend to j, very negative elsewhere
```

*Build the block-causal mask: a token sees the state and its own question, never a sibling.*

Token i may attend to token j when j is not after i, and j is in the state or in the same
question as i. The mask is added to the attention scores, so a blocked pair gets the most
negative number the type can hold.

## A small request

Take three context tokens, then two questions of two tokens each. `encode` gives every token a segment:
0 for the context, 1 for the first question, 2 for the second. In the grid, each row is a token, and a
filled cell means it may attend to the token in that column.

In [ ]:
def show(seg, mask=branch_mask):
    allow = mask(seg, "cpu")[0, 0] == 0
    print("   " + " ".join(str(s) for s in seg))
    for s, row in zip(seg, allow, strict=True):
        print(f"{s}  " + " ".join("#" if ok else "." for ok in row))


show([0, 0, 0, 1, 1, 2, 2])

   0 0 0 1 1 2 2
0  # . . . . . .
0  # # . . . . .
0  # # # . . . .
1  # # # # . . .
1  # # # # # . .
2  # # # . . # .
2  # # # . . # #


Read the last two rows. The second question sees the three context tokens and itself. The two columns
of the first question are empty: to the second question, the first one is not there.

## The rule

Token *i* may attend to token *j* when *j* is not later than *i*, and *j* is in the context or in the
same question as *i*. Nothing else. A longer request shows the same blocks:

In [ ]:
show([0, 0, 1, 1, 1, 2, 3, 3])

   0 0 1 1 1 2 3 3
0  # . . . . . . .
0  # # . . . . . .
1  # # # . . . . .
1  # # # # . . . .
1  # # # # # . . .
2  # # . . . # . .
3  # # . . . . # .
3  # # . . . . # #


## The mask as it was at step 05

The model arrived with a plain causal mask, the usual one for a language model: every token sees
everything before it. Here is the same request under that mask. The second question's rows now have
the first question's columns filled: it reads the first question as if it were part of the context.
Nothing crashes, the model still returns probabilities, and the answers to one question depend on
which questions were asked before it. Only a test of the rule above can tell.

In [ ]:
import torch


def plain_causal_mask(seg, device, dtype=torch.float32):
    n = len(seg)
    causal = torch.tril(torch.ones(n, n, dtype=torch.bool, device=device))
    blocked = torch.finfo(dtype).min
    return torch.zeros(n, n, dtype=dtype, device=device).masked_fill(~causal, blocked)[None, None]


show([0, 0, 0, 1, 1, 2, 2], plain_causal_mask)

   0 0 0 1 1 2 2
0  # . . . . . .
0  # # . . . . .
0  # # # . . . .
1  # # # # . . .
1  # # # # # . .
2  # # # # # # .
2  # # # # # # #


## Why it matters

Because of this mask, three questions packed into one request get exactly the answers they would get
in three separate requests. That is a property of the mask and not of the weights, so the tests check
it on a tiny model with random weights. The other half of the trick is in `encode`: each question's
positions restart right after the context, so a question looks the same to the model whether it comes
first or last. `PointerHead` then reads the answer off the question's own tokens, and `DecisionModel`
puts the pieces together.